# v0.10.0 -- `upsert()`, `update_or_create()`, `get_or_create()`

Three insert-or-update helpers, all with **identical behaviour on SurrealDB 2.6.x and 3.x**:

- **`model.upsert()`** -- insert-or-**replace** by explicit id (native `UPSERT ... CONTENT`).
  Fields omitted from the model are dropped on replace -- use `merge()` for partial updates.
- **`QuerySet.update_or_create(defaults=, **criteria)`** -- Django-style; returns
  `(instance, created)`. On update it **merges** (untouched fields survive).
- **`QuerySet.get_or_create(defaults=, **criteria)`** -- returns an existing match untouched,
  or creates it.

Writes route through `save()` / `merge()`, so lifecycle signals fire and the primary key anchors
identity.

## 1. Connect

In [1]:
import os
from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
# WebSocket (.../rpc) lets transaction() use SurrealDB 3.x native interactive transactions.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root", password="root",
    namespace="examples", database="examples",
)
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 2. Models and reset

In [2]:
import contextlib

from surreal_orm_lite import BaseSurrealModel, SurrealConfigDict


class Customer(BaseSurrealModel):
    model_config = SurrealConfigDict(primary_key="id")
    id: str | None = None
    name: str
    age: int = 0


class Profile(BaseSurrealModel):
    """Extra fields outside the usual criteria/defaults, so REPLACE vs MERGE is observable."""
    model_config = SurrealConfigDict(primary_key="id")
    id: str | None = None
    name: str
    email: str = ""
    role: str = "user"
    age: int = 0


client = await SurrealDBConnectionManager.get_client()
for table in ("Customer", "Profile"):
    with contextlib.suppress(Exception):
        await client.query(f"DELETE {table};", {})
print("tables reset")

tables reset


## 3. `upsert()` -- create, then **replace**

The second `upsert` omits `age`, so REPLACE drops the old value (back to the default `0`). There is still only one record.

In [3]:
await Customer(id="alice", name="Alice", age=30).upsert()
print("after create:", await client.query("SELECT name, age FROM Customer:alice;", {}))

await Customer(id="alice", name="Alice").upsert()  # age omitted -> REPLACE drops it
print("after replace:", await client.query("SELECT name, age FROM Customer:alice;", {}))
print("record count:", len(await client.query("SELECT * FROM Customer;", {})))

after create: [{'age': 30, 'name': 'Alice'}]
after replace: [{'age': 0, 'name': 'Alice'}]
record count: 1


## 4. `update_or_create()` -- create then update, returning `(instance, created)`

In [4]:
obj, created = await Customer.objects().update_or_create(name="Bob", defaults={"age": 1})
print("first call:", obj.name, obj.age, "created:", created)

obj, created = await Customer.objects().update_or_create(name="Bob", defaults={"age": 2})
print("second call:", obj.name, obj.age, "created:", created)
print("Bob count:", len(await client.query("SELECT * FROM Customer WHERE name = 'Bob';", {})))

first call: Bob 1 created: True
second call: Bob 2 created: False
Bob count: 1


## 5. The update path **merges** -- untouched fields survive

Updating `Profile`'s `age` must not wipe `email`/`role`, unlike `upsert()`'s REPLACE.

In [5]:
await Profile(id="p1", name="Carol", email="carol@x.io", role="admin", age=5).save()

obj, created = await Profile.objects().update_or_create(name="Carol", defaults={"age": 9})
print("created:", created, "| age:", obj.age, "| email kept:", obj.email, "| role kept:", obj.role)

created: False | age: 9 | email kept: carol@x.io | role kept: admin


## 6. `get_or_create()` -- never overwrites an existing match

Even though the second call passes a different default, the stored record is returned unchanged.

In [6]:
first, c1 = await Customer.objects().get_or_create(name="Zoe", defaults={"age": 7})
print("first:", first.age, "created:", c1)

second, c2 = await Customer.objects().get_or_create(name="Zoe", defaults={"age": 99})
print("second:", second.age, "created:", c2)  # age stays 7

first: 7 created: True
second: 7 created: False


## 7. Guard rails

Both helpers need at least one lookup criterion, and refuse to act when the criteria match more than one record (which one would you update?).

In [7]:
from surreal_orm_lite import SurrealDbError

try:
    await Customer.objects().update_or_create(defaults={"age": 1})  # no criteria
except SurrealDbError as exc:
    print("no criteria:", exc)

await Customer(id="d1", name="Dup", age=1).save()
await Customer(id="d2", name="Dup", age=2).save()
try:
    await Customer.objects().get_or_create(name="Dup")  # two matches
except SurrealDbError as exc:
    print("multiple matches:", exc)

no criteria: update_or_create() requires at least one lookup criteria.
multiple matches: get_or_create() matched multiple records; the lookup criteria are not unique.


## 8. `upsert()` inside a transaction

Like the other writes, `upsert(tx=)` enrols in a transaction and rolls back with it.

In [8]:
with contextlib.suppress(Exception):
    await client.query("DELETE Customer;", {})

async with SurrealDBConnectionManager.transaction() as tx:
    await Customer(id="txu", name="Tx", age=1).upsert(tx=tx)
print("committed upsert:", await client.query("SELECT name FROM Customer:txu;", {}))

committed upsert: [{'name': 'Tx'}]


## 9. Cleanup

In [9]:
for table in ("Customer", "Profile"):
    await client.query(f"DELETE {table};", {})
await SurrealDBConnectionManager.close_connection()
print("done")

done
